# CardioIA Fase 2 | Parte 1: Extração de sintomas e sugestão de diagnóstico

A ideia aqui é simular o primeiro passo de um sistema de apoio ao diagnóstico:

1. Ler relatos curtos de pacientes (`sintomas.txt`)
2. Encontrar nesses relatos expressões que estão no nosso **mapa de conhecimento** (`mapa_conhecimento.csv`)
3. Somar as "pistas" de cada doença e sugerir a mais provável

É NLP baseado em regras (Cap. 10 da fase): nada de modelo treinado, só texto bem organizado + lógica.

> Aviso: projeto acadêmico. As associações são simplificadas e não servem para diagnóstico real.

**Rodando no Google Colab?** A célula abaixo baixa os arquivos do repositório. Rodando local, ela não faz nada.

In [ ]:
import os
if not os.path.exists("sintomas.txt"):
    !git clone -q https://github.com/pedromoreira427/cardioia-fase2.git
    %cd cardioia-fase2/parte1

In [1]:
import re
import unicodedata
from collections import Counter, defaultdict

import pandas as pd

## 1. Lendo os arquivos

In [2]:
with open("sintomas.txt", encoding="utf-8") as f:
    frases = [linha.strip() for linha in f if linha.strip()]

mapa = pd.read_csv("mapa_conhecimento.csv")

print(f"{len(frases)} frases carregadas")
print(f"{len(mapa)} linhas no mapa de conhecimento, {mapa['doenca_associada'].nunique()} doenças")
mapa.head(10)

10 frases carregadas
34 linhas no mapa de conhecimento, 7 doenças


,sintoma_1,sintoma_2,doenca_associada
0,dor no peito,aperto no torax,Infarto Agudo do Miocardio
1,aperto no peito,pressao no peito,Infarto Agudo do Miocardio
2,braco esquerdo,dor que irradia,Infarto Agudo do Miocardio
3,suor frio,sudorese,Infarto Agudo do Miocardio
4,enjoo,nausea,Infarto Agudo do Miocardio
5,dor na mandibula,dor esmagadora,Infarto Agudo do Miocardio
6,esforco fisico,piora com esforco,Angina
7,passa quando eu descanso,melhora com repouso,Angina
8,dor ao subir escada,dor ao caminhar,Angina
9,queimacao no peito,peso no peito,Angina


## 2. Normalização do texto

Paciente escreve "tórax", "torax", "TÓRAX"... Para o computador são palavras diferentes.
Então antes de comparar, deixo tudo minúsculo e sem acento. A mesma função é usada no mapa e nas frases, senão não bate.

In [3]:
def normalizar(texto: str) -> str:
    texto = texto.lower()
    # separa a letra do acento e joga o acento fora
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

normalizar("Sinto um APERTO no Tórax")

'sinto um aperto no torax'

## 3. Montando a ontologia (doença → expressões)

O CSV tem o formato pedido (`sintoma_1 | sintoma_2 | doenca_associada`). Para buscar fica mais fácil virar um dicionário onde cada doença aponta para todas as expressões ligadas a ela.

Algumas expressões são "radicais" de propósito: `palpitac` pega *palpitação* e *palpitações*, `inchad` pega *inchado* e *inchada*.

In [4]:
ontologia = defaultdict(set)
for _, linha in mapa.iterrows():
    for col in ["sintoma_1", "sintoma_2"]:
        if pd.notna(linha[col]):
            ontologia[linha["doenca_associada"]].add(normalizar(linha[col]))

for doenca, exprs in ontologia.items():
    print(f"{doenca:28} -> {sorted(exprs)}")

Infarto Agudo do Miocardio   -> ['aperto no peito', 'aperto no torax', 'braco esquerdo', 'dor esmagadora', 'dor na mandibula', 'dor no peito', 'dor que irradia', 'enjoo', 'nausea', 'pressao no peito', 'sudorese', 'suor frio']
Angina                       -> ['dor ao caminhar', 'dor ao subir escada', 'esforco fisico', 'melhora com repouso', 'passa quando eu descanso', 'peso no peito', 'piora com esforco', 'queimacao no peito']
Insuficiencia Cardiaca       -> ['acordo a noite', 'cansaco constante', 'cansaco extremo', 'cansado o tempo todo', 'dificuldade para respirar', 'dois travesseiros', 'engordei rapido', 'fadiga', 'falta de ar', 'ganho de peso', 'inchaco', 'inchad', 'sem ar', 'sem folego', 'tosse a noite', 'tosse seca']
Arritmia                     -> ['batimento descompassado', 'batimentos irregulares', 'coracao acelerado', 'coracao dispara', 'coracao falhando', 'coracao pulando', 'palpitac', 'taquicardia']
Hipertensao Arterial         -> ['cabeca latejando', 'dor de cabeca na nuca'

## 4. Extraindo sintomas de uma frase

Uso regex com `\b` (borda de palavra) no começo da expressão. Assim `sem ar` não casa dentro de "sem armário", por exemplo.

Também tem um tratamento simples de **negação**: se aparecer "não", "nunca" ou "sem" logo antes do sintoma (até 3 palavras), ele é ignorado. Sem isso, "não sinto dor no peito" viraria suspeita de infarto.

In [5]:
NEGACOES = {"nao", "nunca", "nem", "jamais"}

def esta_negado(texto: str, inicio: int) -> bool:
    palavras_antes = texto[:inicio].split()[-3:]
    return any(p in NEGACOES for p in palavras_antes)

def extrair_sintomas(frase: str):
    """Retorna lista de (expressao_encontrada, doenca)."""
    texto = normalizar(frase)
    achados = []
    for doenca, exprs in ontologia.items():
        for expr in exprs:
            for m in re.finditer(r"\b" + re.escape(expr), texto):
                if not esta_negado(texto, m.start()):
                    achados.append((expr, doenca))
                    break  # conta cada expressão uma vez só
    return achados

extrair_sintomas(frases[0])

[('dor no peito', 'Infarto Agudo do Miocardio'),
 ('esforco fisico', 'Angina'),
 ('passa quando eu descanso', 'Angina')]

## 5. Sugerindo o diagnóstico

Cada expressão encontrada é um "voto" para a doença. A doença com mais votos é a sugestão principal.
Também mostro uma **confiança** simples: votos da vencedora ÷ total de votos. Se ficou dividido, a confiança cai e isso é sinal para o médico olhar com mais cuidado.

In [6]:
def sugerir_diagnostico(frase: str):
    achados = extrair_sintomas(frase)
    if not achados:
        return {"sintomas": [], "diagnostico": "Sem correspondência no mapa", "confianca": 0.0, "alternativas": []}

    votos = Counter(doenca for _, doenca in achados)
    ranking = votos.most_common()
    principal, n = ranking[0]
    return {
        "sintomas": [expr for expr, _ in achados],
        "diagnostico": principal,
        "confianca": round(n / sum(votos.values()), 2),
        "alternativas": [d for d, _ in ranking[1:]],
    }

## 6. Rodando nas 10 frases

In [7]:
resultados = []
for i, frase in enumerate(frases, start=1):
    r = sugerir_diagnostico(frase)
    print(f"Paciente {i}: {frase}")
    print(f"  sintomas identificados: {r['sintomas']}")
    print(f"  diagnóstico sugerido:   {r['diagnostico']} (confiança {r['confianca']:.0%})")
    if r["alternativas"]:
        print(f"  também considerar:      {', '.join(r['alternativas'])}")
    print()
    resultados.append({"paciente": i, "frase": frase, **r})

Paciente 1: Há dois dias estou com uma dor no peito que piora quando faço esforço físico e passa quando eu descanso.
  sintomas identificados: ['dor no peito', 'esforco fisico', 'passa quando eu descanso']
  diagnóstico sugerido:   Angina (confiança 67%)
  também considerar:      Infarto Agudo do Miocardio

Paciente 2: Sinto cansaço constante há uma semana, mesmo depois de descansar, e meus tornozelos ficam inchados no fim do dia.
  sintomas identificados: ['cansaco constante', 'inchad']
  diagnóstico sugerido:   Insuficiencia Cardiaca (confiança 100%)

Paciente 3: Desde ontem à noite sinto um aperto no tórax que desce para o braço esquerdo, com suor frio e enjoo.
  sintomas identificados: ['aperto no torax', 'braco esquerdo', 'enjoo', 'suor frio']
  diagnóstico sugerido:   Infarto Agudo do Miocardio (confiança 100%)

Paciente 4: Há umas três semanas meu coração dispara do nada e fico com palpitações até deitado vendo TV.
  sintomas identificados: ['coracao dispara', 'palpitac']
  diag

In [8]:
df = pd.DataFrame(resultados)
df["sintomas"] = df["sintomas"].apply(", ".join)
df["alternativas"] = df["alternativas"].apply(", ".join)
df.to_csv("resultados_parte1.csv", index=False, encoding="utf-8")
df[["paciente", "sintomas", "diagnostico", "confianca", "alternativas"]]

,paciente,sintomas,diagnostico,confianca,alternativas
0,1,"dor no peito, esforco fisico, passa quando eu ...",Angina,0.67,Infarto Agudo do Miocardio
1,2,"cansaco constante, inchad",Insuficiencia Cardiaca,1.00,
2,3,"aperto no torax, braco esquerdo, enjoo, suor frio",Infarto Agudo do Miocardio,1.00,
3,4,"coracao dispara, palpitac",Arritmia,1.00,
4,5,"sem folego, cansado o tempo todo",Insuficiencia Cardiaca,1.00,
5,6,"levantei, quase desmaiei, tontura",Sincope / Hipotensao,1.00,
6,7,"dor de cabeca na nuca, pressao anda alta",Hipertensao Arterial,1.00,
7,8,"dor no peito, piora quando respiro fundo, incl...",Pericardite,0.75,Infarto Agudo do Miocardio
8,9,"acordo a noite, sem ar, inchaco, dois travesse...",Insuficiencia Cardiaca,1.00,
9,10,"coracao falhando, batimentos irregulares, vist...",Arritmia,0.67,Sincope / Hipotensao


## 7. Testando os limites

Bom sistema é aquele que a gente sabe onde quebra. Alguns testes:

In [9]:
testes = [
    "Não sinto dor no peito, só um cansaço constante",       # negação
    "Meu peito tá doendo muito desde cedo",                     # sinônimo que não está no mapa
    "Tô com uma dor estranha na barriga",                       # fora do domínio cardio
]
for t in testes:
    r = sugerir_diagnostico(t)
    print(f"{t}\n  -> {r['diagnostico']} | sintomas: {r['sintomas']}\n")

Não sinto dor no peito, só um cansaço constante
  -> Insuficiencia Cardiaca | sintomas: ['cansaco constante']

Meu peito tá doendo muito desde cedo
  -> Sem correspondência no mapa | sintomas: []

Tô com uma dor estranha na barriga
  -> Sem correspondência no mapa | sintomas: []



**O que dá pra ver:**

- A negação funciona no caso simples ("não sinto dor no peito" não conta como dor no peito).
- "Meu peito tá doendo" não é reconhecido, porque o mapa só tem "dor no peito". Regras só enxergam o que foi escrito nelas. Quanto mais linhas no mapa, melhor, mas nunca cobre tudo. Esse é o motivo da Parte 2 usar Machine Learning.
- Fora do domínio o sistema não inventa: devolve "sem correspondência", o que é o comportamento certo.

**Limitações e governança:**

- O mapa foi montado por mim, com base em material de estudo, não validado por médico.
- Um mesmo sintoma (ex.: dor no peito) aparece em várias doenças; o sistema resolve por contagem, que é uma heurística bem simples.
- Isso aqui é **apoio**, nunca substitui avaliação clínica.